# 综合工程实践

学习目标：制作并验证一个接收外部读数、保留泛型关系、支持异步来源且可分发类型声明的小模块。

前置知识：运行时守卫、泛型、可辨识联合、异步错误处理、自动化测试与本地包分发。

适用版本与条件：TypeScript 7.0.2、Node.js 24.11.0；使用 ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/25-data-package/。

1. [package/src/index.ts](scripts/25-data-package/package/src/index.ts)：完整数据处理模块。
2. [readings.test.ts](scripts/25-data-package/readings.test.ts)：有效、无效、边界及异步失败测试。
3. [type-tests.ts](scripts/25-data-package/type-tests.ts)：泛型结果关系的静态测试。
4. [package/package.json](scripts/25-data-package/package/package.json)：独立模块的导出契约。
5. [consumer/](scripts/25-data-package/consumer/)：只通过安装包调用的消费者。
6. [pack-check.mjs](scripts/25-data-package/pack-check.mjs)：本章独立 tarball 安装实验。
7. [tsconfig.json](scripts/25-data-package/tsconfig.json)：声明构建；测试配置为 tsconfig.test.json。

## 1 约定输入、输出和模块边界

模块接受来源未知的读数数组，每项有非空传感器名称和有限数值。名称去掉首尾空白，空数组是有效结果；重复名称允许出现，不在本例自动合并。返回值把失败原因与成功数组分开，便于调用者明确处理。

Result&lt;T&gt; 的 T 表示成功分支的值类型，不描述错误类型。模块只依赖标准语言能力；Node.js 只作为测试和消费宿主，不进入公共类型接口。

以下片段来自 package/src/index.ts。

```typescript
export interface Reading { sensor: string; value: number }
export type Result<T> = { ok: true; value: T }
  | { ok: false; code: "input" | "source"; message: string };
```

Step 1：检查本章正常项目。

```bash
npm run check:25
# 无类型诊断。
```

Step 2：生成当前源码的 JavaScript。

```bash
npm run build:25
# 类型错误时不生成新输出。
```

Step 3：执行本章运行入口。

```bash
npm run run:25
# 执行三项测试，再输出 installed readings OK 和 pack 25 OK。
```

## 2 在入口校验并生成规范数据

本节把“结构校验与数据规范化”本身作为练习目标，特意比较合法、非法和稀疏输入，因此保留这些判断；可分发模块的身份本身并不要求其他教学代码增加输入保护。

isReading 只核对非 null 对象或函数是否具有字符串 sensor 和数值 value；带相应字段的数组、函数或包含 NaN 的对象也符合这一结构。parseReadings 再按业务拒绝数组元素、函数元素、空名称与非有限数值。不能让结构谓词的 false 分支错误地排除本来满足 Reading 的值。

parseReadings 先检查顶层数组，再用 unknown[] 接住元素，不沿用 Array.isArray 收窄后可能过宽的元素类型。for...of 会让稀疏位置作为 undefined 接受检查，防止只用 every 时跳过空槽造成误判。

通过后重新构造 Reading，只保留两个公开字段并规范化名称。源对象的额外属性和对象身份不会直接传出；这与“守卫只缩小静态类型”是不同的运行时操作。NaN、Infinity 由 Number.isFinite 拒绝，零与负数在本例允许。

```typescript
export function isReading(value: unknown): value is Reading {
  return (typeof value === "object" && value !== null || typeof value === "function")
    && "sensor" in value && typeof value.sensor === "string"
    && "value" in value && typeof value.value === "number";
}
export function parseReadings(value: unknown): Result<Reading[]> {
  if (!Array.isArray(value)) return { ok: false, code: "input", message: "需要数组" };
  const items: unknown[] = value;
  const readings: Reading[] = [];
  for (const item of items) {
    if (!isReading(item)) return { ok: false, code: "input", message: "读数无效" };
    if (typeof item === "function" || Array.isArray(item) || item.sensor.trim().length === 0
      || !Number.isFinite(item.value)) return { ok: false, code: "input", message: "读数无效" };
    readings.push({ sensor: item.sensor.trim(), value: item.value });
  }
  return { ok: true, value: readings };
}
```

## 3 泛型函数保留选择结果

mapValues 的 T 表示输入元素类型，U 表示选择函数的返回类型。select 接收 T，返回 U，因此函数整体返回 U[]。readonly 输入约束本函数不直接改写数组，不表示深冻结。

这里仅对已通过入口校验的数组执行映射。泛型不承担运行时校验，也不能保证来自未检查 JavaScript 的数组没有空槽；边界校验和泛型签名承担不同职责。

```typescript
export function mapValues<T, U>(items: readonly T[], select: (item: T) => U): U[] {
  return items.map(select);
}
```

以下片段来自 type-tests.ts。

```typescript
import { mapValues, type Reading } from "./package/src/index.js";
const values: number[] = mapValues<Reading, number>([{ sensor: "a", value: 2 }], item => item.value);
// @ts-expect-error 回调返回 number，不能把结果当 string[]。
const texts: string[] = mapValues([1], item => item + 1);
void values;
void texts;
```

## 4 异步来源和错误处理

source 是一个返回 Promise&lt;unknown&gt; 的函数。通过参数注入来源，测试可以提供立即完成、拒绝或无效数据，不必真实访问网络。

try 只处理来源的调用与 await，拒绝值为 Error 时读取 message，否则给出稳定说明。数据结构失败交给 parseReadings，并保留 input 与 source 两类错误；不能把来源成功等同于数据合法。

```typescript
export async function loadReadings(source: () => Promise<unknown>): Promise<Result<Reading[]>> {
  let input: unknown;
  try { input = await source(); }
  catch (error: unknown) {
    return { ok: false, code: "source", message: error instanceof Error ? error.message : "非 Error 异常" };
  }
  return parseReadings(input);
}
```

## 5 同时验证契约和行为

类型测试确认回调返回值推断成 number[]；原始类型反例单独检查 TS2322。运行测试确认输入规范化、空数组、稀疏输入和非有限数值，以及 Promise 的成功、Error 拒绝和非 Error 拒绝。

业务测试使用各自创建的输入，并等待所有异步断言。这里的测试不声称覆盖任意带 getter、代理或恶意副作用的 JavaScript 对象；输入协议面向解析后的普通数据。

以下片段来自 readings.test.ts。

```typescript
import test from "node:test";
import assert from "node:assert/strict";
import { isReading, parseReadings, mapValues, loadReadings } from "./package/src/index.js";
// 第一组确认有效输入、空结果和去除首尾空格后的公开返回值。
test("有效、空数组及规范化", () => {
  assert.deepEqual(parseReadings([]), { ok: true, value: [] });
  assert.deepEqual(parseReadings([{ sensor: " a ", value: 0, extra: true }]),
    { ok: true, value: [{ sensor: "a", value: 0 }] });
  assert.deepEqual(mapValues([{ value: 2 }], item => item.value * 2), [4]);
});
test("无效和稀疏输入", () => {
  // 带同名字段的值可满足结构谓词，但仍可能不满足普通数据的业务约定。
  for (const item of [Object.assign([], { sensor: "a", value: 1 }),
    Object.assign(() => {}, { sensor: "a", value: 1 }), { sensor: " ", value: NaN }]) {
    assert.equal(isReading(item), true);
    assert.equal(parseReadings([item]).ok, false);
  }
  // 稀疏数组含空位；与显式非法元素一起检查解析入口。
  for (const value of [null, {}, [null], Array(1), [{ sensor: "", value: 1 }],
    [{ sensor: "a", value: NaN }], [{ sensor: "a", value: Infinity }], [{ sensor: "a", value: "2" }]]) {
    assert.equal(parseReadings(value).ok, false);
  }
});
// 来源成功、Error 拒绝和非 Error 拒绝分开观察；所有异步结果都要等待。
test("异步来源与任意拒绝值", async () => {
  assert.deepEqual(await loadReadings(async () => []), { ok: true, value: [] });
  assert.deepEqual(await loadReadings(async () => { throw new Error("离线"); }),
    { ok: false, code: "source", message: "离线" });
  assert.deepEqual(await loadReadings(async () => { throw "停止"; }),
    { ok: false, code: "source", message: "非 Error 异常" });
  assert.equal((await loadReadings(async () => [{}])).ok, false);
});
```

以下片段来自 tsconfig.test.json。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": {
    "rootDir": ".",
    "outDir": ".build-test",
    "types": [
      "node"
    ],
    "declaration": false,
    "declarationMap": false
  },
  "files": [
    "package/src/index.ts",
    "readings.test.ts",
    "type-tests.ts"
  ]
}
```

Step 1：单独核对原始泛型错误。

```bash
npm run errors:25
# 退出 1，包含 TS2322。
```

## 6 构建声明与分发模块

包配置只开放根入口，types 与 default 指向同一 dist 的声明和 JavaScript。构建配置启用 declaration 与 declarationMap；测试另用 tsconfig.test.json，测试文件不会进入 package/dist。

files 中保留 src 是为了让声明映射找到源码，消费者不应从 src 导入。工具依赖仍在课程根，发布包没有不必要的 Node 类型依赖。

以下片段来自 tsconfig.json。

```json
{
  "compilerOptions": {
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "strict": true,
    "lib": [
      "ES2025"
    ],
    "types": [],
    "rootDir": "package/src",
    "outDir": "package/dist",
    "noEmitOnError": true,
    "declaration": true,
    "declarationMap": true,
    "noUncheckedIndexedAccess": true
  },
  "files": [
    "package/src/index.ts"
  ]
}
```

以下片段来自 package/package.json。

```json
{
  "name": "notebook-readings-ts-c",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "types": "./dist/index.d.ts",
  "files": [
    "dist",
    "src"
  ],
  "exports": {
    ".": {
      "types": "./dist/index.d.ts",
      "default": "./dist/index.js"
    }
  }
}
```

## 7 在独立消费者中闭合流程

消费者只按包名导入公开函数和类型，分别展示 Result 的成功与失败结果；当前成功输入明确只有一条读数，因此直接读取第一项。下面既能检查类型，也会打印规范化后的真实运行值；完整实验驱动再核对输出。

以下片段来自 consumer/main.ts。

```typescript
import { loadReadings, mapValues, type Reading } from "notebook-readings-ts-c";
const result = await loadReadings(async () => [{ sensor: " 温度 ", value: 18 }]);
if (result.ok) {
  const rows: Reading[] = result.value;
  const values: number[] = mapValues(rows, row => row.value);
  console.log("installed readings OK", rows[0].sensor, values[0]); // → installed readings OK 温度 18
} else {
  console.log("读取失败", result.code, result.message); // 练习中来源拒绝时显示 source 与拒绝原因。
}
```

pack-check.mjs 依次构建、打包、复制本章 consumer 并从 tarball 安装。随后检查安装包的 dist/index.d.ts 被载入，打印运行入口并核对消费者输出；命令失败直接抛出，finally 只清理本次临时目录。下面保留消费者的类型反例，供修改公开类型时单独观察 TS2322。

以下片段来自 consumer/type-errors.ts。

```typescript
import { mapValues } from "notebook-readings-ts-c";
const texts: string[] = mapValues([1], item => item + 1);
```

Step 1：只执行本章独立安装消费者实验。

```bash
npm run pack:25
# 输出 node_modules/notebook-readings-ts-c/dist/index.js 的实际位置与 pack 25 OK，随后清理临时目录。
```

这个实验使用本章独立包和消费者。真实业务模块与消费者样例保存在 scripts/25-data-package，只有该次安装产物位于临时目录。

单独检查消费者反例（选学）：正常打包实验不会运行 consumer/type-errors.ts，且结束时会删除临时安装。要观察它对已安装声明的诊断，先完成前文 npm run build:25，再在本章工作目录的 PowerShell 中按下面步骤准备一份独立消费者。开始时 .consumer-errors 目录不存在；安装、缓存与后续清理只涉及这份副本。

Step 1：复制本章消费者样例。

```powershell
Copy-Item scripts/25-data-package/consumer scripts/25-data-package/.consumer-errors -Recurse
```

Step 2：将已构建的本地包安装到副本中。

```powershell
npm install --prefix scripts/25-data-package/.consumer-errors ./scripts/25-data-package/package --install-links --offline --ignore-scripts --no-audit --no-fund --package-lock=false --cache scripts/25-data-package/.consumer-errors/.npm-cache
# --install-links 将本地目录打包后安装为副本；本例安装 1 个包，不创建源码链接。
```

Step 3：只检查副本中的类型反例。

```powershell
node node_modules/typescript/bin/tsc -p scripts/25-data-package/.consumer-errors/tsconfig.errors.json
# 预期退出码 1，只报告 TS2322：number[] 不能赋给 string[]；noEmit 不生成 JavaScript。
```

Step 4：检查结束后删除本次消费者副本及其安装、缓存。

```powershell
Remove-Item -LiteralPath scripts/25-data-package/.consumer-errors -Recurse
```

## 本章小结

- 入口校验建立真实数据依据，泛型保留已经建立的类型关系。
- 结果联合区分结构失败和来源失败，async 接口需要等待来源完成。
- 源码测试、声明生成和独立安装消费者共同验证分发边界。

## 练习

1. 增加汇总平均值接口并定义空数组行为；至少测试空数组、单项和两项，声明与安装消费者都更新。
2. 限制传感器名称长度为 1–20；检查长度 1、20、21 以及纯空白输入。
3. 让消费者传入会拒绝的 source，确认返回 source 分支并输出“读取失败 source 离线”；同步调整打包驱动的输出断言，进程正常结束。

### 提示

1. 明确空数组返回值后再实现累加与除法。
2. 长度检查放在规范化后的名称上，不改变结构谓词的含义。
3. 使用 `async () => { throw new Error("离线"); }` 作为来源。

### 参考解析

1. 可约定空数组返回 null，非空数组返回数值平均值；空数组判断属于这项真实功能的两种结果。以 []、[2]、[2, 4] 为例，应分别得到 null、2、3，并更新公开声明与消费者。
2. trim 后检查长度 1–20；1、20 有效，21 和纯空白无效。isReading 只判断字段结构，不能因业务范围变化就把满足 Reading 的值判为不属于该类型。
3. loadReadings 将这次来源拒绝转为 source 结果，消费者 else 分支输出“读取失败 source 离线”。将打包断言也改为这一整行，完成练习后恢复正常来源及断言。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [5.5 / Inferred Type Predicates](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-5.html#inferred-type-predicates) 的 if-and-only-if 语义及手写谓词边界；[Narrowing](https://www.typescriptlang.org/docs/handbook/2/narrowing.html#discriminated-unions)、[Generics](https://www.typescriptlang.org/docs/handbook/2/generics.html)、[More on Functions](https://www.typescriptlang.org/docs/handbook/2/functions.html#other-types-to-know-about)：类型依据、泛型关系和 unknown；[Publishing](https://www.typescriptlang.org/docs/handbook/declaration-files/publishing.html)、[declarationMap](https://www.typescriptlang.org/tsconfig/declarationMap.html)：公共包契约。 |
| ECMA-262 第 16 版 | [Number.isFinite](https://tc39.es/ecma262/2025/multipage/numbers-and-dates.html#sec-number.isfinite)；[Array.prototype.values 与 CreateArrayIterator](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-createarrayiterator)、同页 Array.prototype.every：有限数值及稀疏数组访问边界。 |
| Node.js 24.11.0 | [fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[fsPromises.cp](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisescpsrc-dest-options)：递归复制及等待完成；[测试运行器](https://nodejs.org/download/release/v24.11.0/docs/api/test.html)、[断言](https://nodejs.org/download/release/v24.11.0/docs/api/assert.html)、[execFileSync](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#child_processexecfilesyncfile-args-options) 的非零退出异常、[包入口](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#package-entry-points)：运行测试与真实包消费。 |
| npm 11 | [npm pack](https://docs.npmjs.com/cli/v11/commands/npm-pack/)、[npm install](https://docs.npmjs.com/cli/v11/commands/npm-install/#install-links)：离线本地 tarball 安装。 |
| Microsoft Learn | [Copy-Item：复制目录](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.management/copy-item?view=powershell-7.5#example-3-copy-directory-and-contents-to-a-new-directory)、[Remove-Item / Recurse](https://learn.microsoft.com/en-us/powershell/module/microsoft.powershell.management/remove-item?view=powershell-7.5#-recurse)：PowerShell 中建立与清理本次消费者副本。 |